In [1]:
from pathlib import Path
import pandas as pd

from transformers import AutoTokenizer, AutoModel
import joblib

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import torch.nn as nn

START_YEAR = 2019

In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

In [3]:
class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [4]:
@torch.no_grad()
def embed(texts, batch_size=32):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt').to(device)
            output = model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu())
    return torch.cat(embeddings)

**NECESSARY DETAILS (To Maintain):** 
1. ~~Fiscal Year~~ ( Competition FY )
2. ~~Start/End Date~~
3. Insitution Name
4. ~~Department/Faculty~~
5. Total Award amount
6. Funding Program ( name and type )
7. Research Title (may not be needed)
8. Field of Research
9. Keywords about Research
10. Funding Reference Numbers

**Issues**

1. FYs does not seem to provide much information as grants are given over a number of FYs.
2. Only CIHR has START/END dates; This seems to be replacable by Competiton date.
4. Department/Faculty, CIHR seems to have a lot of NaN and SSHRC does not.

**Data Cleaning for CIHR**

In [5]:
cihr_path = "raw_data/CIHR/"
cihr_files = Path(cihr_path).glob("*.csv")

CIHR_DFS = [pd.read_csv(f) for f in cihr_files]
CIHR_DATA = pd.concat(CIHR_DFS, ignore_index=True)

In [6]:
grant_descriptors = [
    "FundingCode_CodeFinancement", "CompetitionFY_AFConcours", #"FundingStartDate_DatePremierVersement", "FundingEndDate_DateDernierVersement", 
    "ResearchInstitutionNameEN_NomEtablissementRechercheAN", "ResearchInstitutionNameFR_NomEtablissementRechercheFR",
    "TotalAmountAwarded_MontantTotalAccorde","ProgramNameEN_NomProgrammeAN", "ProgramTypeEN_TypeProgrammeAN", 
    "ApplicationTitle_TitreDemande", "PrimaryThemeEN_ThemePrincipalAN"
]

col_names = [
    'Unique_ID', 'CompetitionFY', 'Institution', 'Institution_FR',
    'Total_Amount', 'Program_Name', 'Program_Type', 'Title', 'Main_Discipline'
]

CIHR_DATA = CIHR_DATA[grant_descriptors]
CIHR_DATA.columns = col_names

CIHR_DATA.drop_duplicates(inplace=True)
CIHR_DATA.dropna(subset=["Total_Amount"], inplace=True)

# CIHR_DATA_SORTED = CIHR_DATA.sort_values(by="Unique_ID") # FundingCode_CodeFinancement is historical unique identifer
# CIHR_DATA_SORTED.head(8)


In [7]:
CIHR_DATA['CompetitionFY'] = CIHR_DATA['CompetitionFY']//100 # convert to year (ex. 201920 -> 2019)
CIHR_DATA = CIHR_DATA[CIHR_DATA["CompetitionFY"] >= START_YEAR]
# CIHR_DATA.head(5)

*If the Instituion has an FR but no EN name (NaN), need to replace EN w/ the suitable name*

In [8]:
CIHR_DATA['Institution'] = CIHR_DATA['Institution'].fillna(CIHR_DATA['Institution_FR'])
CIHR_DATA.drop(columns=["Institution_FR"], inplace=True)

# CIHR_DATA_SORTED.head(5)

In [9]:
CIHR_DATA['Institution'].value_counts()

Institution
University of Toronto                                                     1392
University of British Columbia                                            1227
McGill University                                                          893
University of Calgary                                                      783
McMaster University                                                        657
                                                                          ... 
Northwest Territories SPOR Support Unit                                      1
Ontario Spor Support Unit (Toronto, Ontario)                                 1
George & Fay Yee Centre for Healthcare Innovation (Winnipeg, Manitoba)       1
Alberta Innovates (Edmonton)                                                 1
International Longevity Centre Canada (Ottawa)                               1
Name: count, Length: 705, dtype: int64

In [ ]:
label_mapping = joblib.load("models/CIHR_MD_label_mapping.pkl")

input_dim = 384  # MiniLM embedding size
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
clf_model.load_state_dict(torch.load("models/CIHR_MD.pt"))
# clf_model.eval()

Classifier(
  (net): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=4, bias=True)
  )
)

In [11]:
# missing_df = CIHR_DATA[CIHR_DATA["Main_Discipline"].isna() | CIHR_DATA["Main_Discipline"] == "Not applicable/Specified"].copy()
CIHR_DATA['Main_Discipline'] = CIHR_DATA['Main_Discipline'].replace(['Not applicable/Specified', 'Not Applicable', ''], None)
missing_df = CIHR_DATA[CIHR_DATA['Main_Discipline'].isna()].copy()

X_missing = embed(missing_df['Title'].tolist())
with torch.no_grad():
    preds = clf_model(X_missing.to(device)).argmax(dim=1).cpu().numpy()
    pred_labels = [label_mapping[i] for i in preds]

CIHR_DATA.loc[missing_df.index, "Main_Discipline"] = pred_labels

**Data Cleaning for NSERC**

In [12]:
nserc_path = "raw_data/NSERC/"
nserc_files = Path(nserc_path).glob("*.csv")

NSERC_DFS = [pd.read_csv(f) for f in nserc_files]
NSERC_DATA = pd.concat(NSERC_DFS, ignore_index=True)

In [13]:
grant_descriptors = [
    "ApplicationID", "CompetitionYear-Année de concours",
    "Institution-Établissement",
    "AwardAmount", "ProgramNameEN", "GroupEN",
    "ApplicationTitle", "AreaOfApplicationGroupEN"
]

col_names = [
    'Unique_ID', 'CompetitionFY', 'Institution', 
    'Total_Amount', 'Program_Name', 'Program_Type', 'Title', 'Main_Discipline'
]

NSERC_DATA = NSERC_DATA[grant_descriptors]
NSERC_DATA.columns = col_names

NSERC_DATA.drop_duplicates(inplace=True)
NSERC_DATA.dropna(subset=["Total_Amount"], inplace=True)

# NSERC_DATA_SORTED = NSERC_DATA.sort_values(by="Unique_ID")
# NSERC_DATA_SORTED.head(5)



In [14]:
NSERC_DATA = NSERC_DATA[NSERC_DATA["CompetitionFY"] >= START_YEAR]
# NSERC_DATA_SORTED.head(20)

In [15]:
# NSERC_DATA['Institution'].value_counts()

In [ ]:
label_mapping = joblib.load("models/NSERC_MD_label_mapping.pkl")

input_dim = 384  # MiniLM embedding size
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
clf_model.load_state_dict(torch.load("models/NSERC_MD.pt"))


<All keys matched successfully>

In [17]:
NSERC_DATA['Main_Discipline'] = NSERC_DATA['Main_Discipline'].replace(['Not available'], None)
missing_df = NSERC_DATA[NSERC_DATA['Main_Discipline'].isna()].copy()

X_missing = embed(missing_df["Title"].tolist())
with torch.no_grad():
    preds = clf_model(X_missing.to(device)).argmax(dim=1).cpu().numpy()
    pred_labels = [label_mapping[i] for i in preds]

NSERC_DATA.loc[missing_df.index, "Main_Discipline"] = pred_labels

**Data Clearning for SSHRC**

In [18]:
sshrc_path = "raw_data/SSHRC/"
sshrc_files = Path(sshrc_path).glob("*.csv")

SSHRC_DFS = [pd.read_csv(f) for f in sshrc_files]
SSHRC_DATA = pd.concat(SSHRC_DFS, ignore_index=True)

In [19]:
grant_descriptors = [
    "cle", "Competition_Year-Année_du_concours",
    "Institution",
    "Amount-Montant", "Program",
    "Title-Titre", "Main_Discipline"
]

col_names = [
    'Unique_ID', 'CompetitionFY', 'Institution',
    'Total_Amount', 'Program_Name', 'Title', 'Main_Discipline'
]


SSHRC_DATA = SSHRC_DATA[grant_descriptors]
SSHRC_DATA.columns = col_names

SSHRC_DATA.drop_duplicates(inplace=True)
SSHRC_DATA.dropna(subset=["Total_Amount"], inplace=True)

# SSHRC_DATA_SORTED = SSHRC_DATA.sort_values(by="Unique_ID")
# SSHRC_DATA_SORTED.head(5)

In [20]:
SSHRC_DATA = SSHRC_DATA[SSHRC_DATA["CompetitionFY"] >= START_YEAR]
# SSHRC_DATA.head(20)

In [21]:
SSHRC_DATA['Institution'].value_counts()

Institution
University of Toronto                                                 2103
The University of British Columbia                                    1395
McGill University                                                     1201
Grantee                                                               1140
York University                                                        996
                                                                      ... 
St. Thomas More College                                                  1
Medicine Hat College                                                     1
Cégep de Rivière-du-Loup                                                 1
Loyalist College (Loyalist College of Applied Arts and Technology)       1
Université de Hearst                                                     1
Name: count, Length: 244, dtype: int64

*Need Consistency in Institution Naming w.r.t. to other datasets*

In [ ]:
label_mapping = joblib.load("models/SSHRC_MD_label_mapping.pkl")

input_dim = 384  # MiniLM embedding size
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
clf_model.load_state_dict(torch.load("models/SSHRC_MD.pt"))
# clf_model.eval()

Classifier(
  (net): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=20, bias=True)
  )
)

In [23]:
SSHRC_DATA['Main_Discipline'] = SSHRC_DATA['Main_Discipline'].replace(["Not Specified", "Not specified", "Not Applicable", "Multiple primary fields of research", "Interdisciplinary Studies"], None)
missing_df = SSHRC_DATA[SSHRC_DATA['Main_Discipline'].isna()].copy()

X_missing = embed(missing_df["Title"].tolist())
with torch.no_grad():
    preds = clf_model(X_missing.to(device)).argmax(dim=1).cpu().numpy()
    pred_labels = [label_mapping[i] for i in preds]

SSHRC_DATA.loc[missing_df.index, "Main_Discipline"] = pred_labels

**CHECKING AGNECY DATA**

In [24]:
CIHR_DATA.to_csv("clean_data/CIHR_DATA.csv", index=False)
NSERC_DATA.to_csv("clean_data/NSERC_DATA.csv", index=False)
SSHRC_DATA.to_csv("clean_data/SSHRC_DATA.csv", index=False)

**COMBINING THE DATASETS**

In [25]:
CIHR_DATA['Agency'] = "CIHR"
NSERC_DATA['Agency'] = "NSERC"
SSHRC_DATA['Agency'] = "SSHRC"

TRIAGENCY_DATA = pd.concat([CIHR_DATA, NSERC_DATA, SSHRC_DATA], ignore_index=True)

cols = list(TRIAGENCY_DATA.columns)
TRIAGENCY_DATA = TRIAGENCY_DATA[cols[-1:] + cols[:-1]]

print(TRIAGENCY_DATA['Institution'].value_counts())
print(TRIAGENCY_DATA['Main_Discipline'].value_counts())

Institution
University of Toronto                                                       8280
McGill University                                                           5148
University of Alberta                                                       4413
University of British Columbia                                              3679
University of Waterloo                                                      3415
                                                                            ... 
Universite de Lyon I (France)                                                  1
Universitat Hamburg (Germany)                                                  1
Helen Keller International - Cambodia                                          1
Centre Integre de sante & serv. sociaux de la Monteregie-Centre (Quebec)       1
Université de Hearst                                                           1
Name: count, Length: 1505, dtype: int64
Main_Discipline
Advancement of knowledge                 

In [26]:
TRIAGENCY_DATA.to_csv("clean_data/TRIAGENCY_DATA.csv", index=False)